# Benchmark MiniCPM-V-4 — 355 keyframe AIC

Model cuoi cung trong bang report. Truoc day khong chay duoc vi registry ghi
loader="auto_causal" (AutoModelForCausalLM) trong khi MiniCPM can AutoModel va
chu ky .chat() rieng: chat(msgs=..., image=..., tokenizer=...).

Da co MiniCpmAdapter (vlm/adapter_minicpm.py), dinh tuyen qua _la_ho_minicpm().

Thu tu cell: chay -> nen/luu -> kiem.

In [ ]:
import os

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import torch

print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHONG CO')
assert torch.cuda.is_available(), 'Chua bat GPU'


In [ ]:
!pip install -q "transformers>=4.51,<5" accelerate bitsandbytes timm sentencepiece
import transformers
print('transformers:', transformers.__version__)

In [ ]:
import subprocess, sys
from pathlib import Path

# Clone tu fork public -- code luon khop ban moi nhat da day len.
REPO = 'https://github.com/lolizabrett-byte/Multimodal-Agentic-Retrieval-Engine.git'
NHANH = 'research/vlm-prompting'
DICH = Path('/kaggle/working/repo')

# Kaggle giu /kaggle/working giua cac version -> "clone neu chua co" se dung code
# cu cua lan chay truoc. Da mat mot luot GPU vi vay. Xoa roi clone lai moi lan.
import shutil
if DICH.exists():
    shutil.rmtree(DICH)
subprocess.run(['git', 'clone', '--depth', '1', '-b', NHANH, REPO, str(DICH)], check=True)

PKG = DICH / 'system1' / 'research' / 'vlm_prompting'
assert PKG.exists(), f'Khong thay code tai {PKG}'
sys.path.insert(0, str(PKG))

for ten in list(sys.modules):
    if ten.startswith(('vlm', 'benchmark_runner', 'checkpoint_utils', 'quality')):
        del sys.modules[ten]
print('Code tai:', PKG)

hash_code = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                           cwd=DICH, capture_output=True, text=True).stdout.strip()
print('Commit:', hash_code)
assert hash_code, 'Khong doc duoc commit hash -- clone that bai'


In [ ]:
ANH_DIR = next(Path('/kaggle/input').glob('**/images'), None)
print('Thu muc anh:', ANH_DIR)
so_anh = len(list(ANH_DIR.glob('*.jpg')))
print('So anh:', so_anh)
assert so_anh >= 100, f'Chi co {so_anh} anh, de bai can >= 100'


In [ ]:
lenh = [
    sys.executable, 'scripts/benchmark_runner.py',
    '--mode', 'mass',
    '--models', 'minicpm-v-4',
    '--backend', 'transformers',
    '--strict', '--restart',
    '--frames-dir', str(ANH_DIR),
    '--out-dir', '/kaggle/working/ket_qua',
]
print('Chay:', ' '.join(lenh))
kq = subprocess.run(lenh, cwd=str(PKG))
print('Ma thoat:', kq.returncode)
# Khong assert -- ket qua phai duoc nen o cell sau truoc khi bat cu thu gi nem loi.

In [ ]:
import shutil
ZIP = shutil.make_archive('/kaggle/working/benchmark-minicpm', 'zip', '/kaggle/working/ket_qua')
print('Da nen:', ZIP)

In [ ]:
import json
ra = Path('/kaggle/working/ket_qua')
for f in sorted(ra.rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to(ra)}  {f.stat().st_size:,} bytes')
ck = ra / 'checkpoint_minicpm-v-4.json'
assert ck.exists(), 'Khong thay checkpoint minicpm -- model khong nap duoc, doc log o tren'
d = json.loads(ck.read_text(encoding='utf-8'))
xong = d.get('da_xong') or {}
ok = sum(1 for v in xong.values() if v.get('thanh_cong'))
raw = sum(1 for v in xong.values() if v.get('raw_text'))
print(f'{len(xong)} anh | JSON hop le {ok} ({ok/max(len(xong),1):.1%}) | co raw_text {raw}')
for v in list(xong.values())[:2]:
    if v.get('raw_text'):
        print('vi du output hong:', v['raw_text'][:300])